In [1]:
import pandas as pd
import re
import numpy as np


In [2]:
affinities  = pd.read_excel( "/FastHome/gyula/PPB-Affinity_two_chain_with_sequences.xlsx")

In [3]:
peptide_affinities = affinities.loc[affinities.loc[:, "Shorter Than 50 aa"] == True]

In [4]:
mask = peptide_affinities["Chain A Length"] < peptide_affinities["Chain B Length"]

swap_pairs = [
    ("Chain A Length",            "Chain B Length"),
    ("Ligand Name",               "Receptor Name"),
    ("Ligand Chains",             "Receptor Chains"),
    ("Chain A (Ligand) ID",       "Chain B (Receptor) ID"),
    ("Chain A Sequence (Ligand)", "Chain B Sequence (Receptor)"),
]

for col_a, col_b in swap_pairs:
    peptide_affinities.loc[mask, [col_a, col_b]] = \
        peptide_affinities.loc[mask, [col_b, col_a]].values

    Now chain A consistantly smaller than chain B! Cool We are ready to train on this data!

In [5]:
peptide_affinities["Target"] = peptide_affinities["Chain B Sequence (Receptor)"]
peptide_affinities["proteina"] = peptide_affinities["Chain A Sequence (Ligand)"]

In [6]:
peptide_affinities

,Unnamed: 0,Source Data Set,Complex ID,PDB,Mutations,Ligand Chains,Receptor Chains,Ligand Name,Receptor Name,KD(M),...,Subgroup,Chain A (Ligand) ID,Chain B (Receptor) ID,Chain A Sequence (Ligand),Chain B Sequence (Receptor),Chain A Length,Chain B Length,Shorter Than 50 aa,Target,proteina
0,0,SKEMPI v2.0,"1A22:A, B::PMID=7504735",1A22,NaN,B,A,hGH binding protein,Human growth hormone,9.000000e-10,...,NaN,B,A,FSGSEATAAILSRAPWSLQSVNPGLKTNSSKEPKFTKCRSPERETF...,FPTIPLSRLFDNAMLRAHRLHQLAFDTYQEFEEAYIPKEQKYSFLQ...,238,191,False,FPTIPLSRLFDNAMLRAHRLHQLAFDTYQEFEEAYIPKEQKYSFLQ...,FSGSEATAAILSRAPWSLQSVNPGLKTNSSKEPKFTKCRSPERETF...
1,1,SKEMPI v2.0,"1A4Y:A, B::PMID=9050852",1A4Y,NaN,A,B,Ribonuclease inhibitor,Angiogenin,5.000000e-16,...,NaN,A,B,SLDIQSLDIQCEELSDARWAELLPLLQQCQVVRLDDCGLTEARCKD...,QDNSRYTHFLTQHYDAKPQGRDDRYCESIMRRRGLTSPCKDINTFI...,460,123,False,QDNSRYTHFLTQHYDAKPQGRDDRYCESIMRRRGLTSPCKDINTFI...,SLDIQSLDIQCEELSDARWAELLPLLQQCQVVRLDDCGLTEARCKD...
2,2,SKEMPI v2.0,"1ACB:E, I::PMID=9048543",1ACB,NaN,E,I,Bovine alpha-chymotrypsin,Eglin c,1.490000e-12,...,NaN,E,I,CGVPAIQPVLSGLSRIVNGEEAVPGSWPWQVSLQDKTGFHFCGGSL...,TEFGSELKSFPEVVGKTVDQAREYFTLHYPQYDVYFLPEGSPVTLD...,245,70,False,TEFGSELKSFPEVVGKTVDQAREYFTLHYPQYDVYFLPEGSPVTLD...,CGVPAIQPVLSGLSRIVNGEEAVPGSWPWQVSLQDKTGFHFCGGSL...
3,4,SKEMPI v2.0,"1AK4:A, D::PMID=9223641",1AK4,NaN,A,D,Cyclophilin A,HIV-1 capsid protein,1.200000e-05,...,NaN,A,D,MVNPTVFFDIAVDGEPLGRVSFELFADKVPKTAENFRALSTGEKGF...,PIVQNLQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEG...,165,145,False,PIVQNLQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEG...,MVNPTVFFDIAVDGEPLGRVSFELFADKVPKTAENFRALSTGEKGF...
4,6,SKEMPI v2.0,"1B2S:A, D::PMID=7739054",1B2S,NaN,A,D,Barnase,Barstar,1.570000e-10,...,NaN,A,D,AQVINTFDGVADYLQTYHKLPDNYITASEAQALGWVASKGNLADVA...,MKKAVINGEQIRSISDLHQTLKKELALPEYYGENLDALWDCLAGWV...,110,90,False,MKKAVINGEQIRSISDLHQTLKKELALPEYYGENLDALWDCLAGWV...,AQVINTFDGVADYLQTYHKLPDNYITASEAQALGWVASKGNLADVA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431,12391,PDBbind v2020,"6THG:I, J::PMID=31862858",6THG,NaN,I,J,"Cedar Virus attachment glycoprotein (G), CedVG",human ephrin-B1,4.000000e-09,...,NaN,I,J,ETGKIFCKSVSKDPDFRLKQIDYVIPVQQDRSICMNNPLLDISDGF...,ETGAKNLEPVSWSSLNPKFLSGKGLVIYPKIGDKLDIICPRAEAGR...,426,151,False,ETGAKNLEPVSWSSLNPKFLSGKGLVIYPKIGDKLDIICPRAEAGR...,ETGKIFCKSVSKDPDFRLKQIDYVIPVQQDRSICMNNPLLDISDGF...
6432,12402,PDBbind v2020,"6UMT:A, B::PMID=31727844",6UMT,NaN,A,B,"Programmed cell death protein 1, human PD-1 (N...","Programmed cell death 1 ligand 2, PD-L2 IgV",2.600000e-09,...,NaN,A,B,MGWSCIILFLVATATGVHSNPPTFSPALLVVTEGDSATFTCSFSST...,MIFLLLMLSLELQLHQIAALFTVTVPKELYIIEHGSDVTLECNFDT...,140,123,False,MIFLLLMLSLELQLHQIAALFTVTVPKELYIIEHGSDVTLECNFDT...,MGWSCIILFLVATATGVHSNPPTFSPALLVVTEGDSATFTCSFSST...
6433,12404,PDBbind v2020,"6UYS:A, B::PMID=31879127",6UYS,NaN,A,B,K37-acetylated SUMO1,phosphorylated PML-SIM,1.600000e-06,...,NaN,A,B,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...,GSGAGEAEERVVVISSSEDSDAENSSSRY,83,29,True,GSGAGEAEERVVVISSSEDSDAENSSSRY,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...
6434,12405,PDBbind v2020,"6UYS:C, D::PMID=31879127",6UYS,NaN,C,D,K37-acetylated SUMO1,phosphorylated PML-SIM,1.600000e-06,...,NaN,C,D,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...,GSGAGEAEERVVVISSSEDSDAENSSSRY,83,29,True,GSGAGEAEERVVVISSSEDSDAENSSSRY,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...


In [7]:

R = 1.987e-3  # kcal/(mol·K)

def parse_temperature(val):
    """Extract first number from a string, return NaN if none found."""
    match = re.search(r"[\d.]+", str(val))
    return float(match.group()) if match else np.nan

def calc_delta_g(row):
    kd = row["KD(M)"]
    T = parse_temperature(row["Temperature(K)"])

    if np.isnan(T) or np.isnan(kd):
        T = 298
    if not (250 <= T <= 310):
        print(f"Error: Temperature {T} K out of range for index {row.name}")
        return np.nan

    return R * T * np.log(kd)  # ΔG = RT·ln(Kd), in kcal/mol

peptide_affinities["Y"] = peptide_affinities.apply(calc_delta_g, axis=1)

Error: Temperature 323.0 K out of range for index 47
Error: Temperature 338.0 K out of range for index 49
Error: Temperature 338.0 K out of range for index 101
Error: Temperature 333.0 K out of range for index 187
Error: Temperature 318.0 K out of range for index 220
Error: Temperature 323.0 K out of range for index 713
Error: Temperature 323.0 K out of range for index 714
Error: Temperature 323.0 K out of range for index 715
Error: Temperature 323.0 K out of range for index 716
Error: Temperature 323.0 K out of range for index 717
Error: Temperature 323.0 K out of range for index 718
Error: Temperature 323.0 K out of range for index 719
Error: Temperature 323.0 K out of range for index 720
Error: Temperature 323.0 K out of range for index 721
Error: Temperature 323.0 K out of range for index 722
Error: Temperature 323.0 K out of range for index 723
Error: Temperature 323.0 K out of range for index 724
Error: Temperature 323.0 K out of range for index 725
Error: Temperature 333.0 K out

In [8]:
peptide_affinities.to_csv("/FastHome/gyula/ppi_data/train_sets/PPI_peptide_experimental.csv")

,Unnamed: 0,Source Data Set,Complex ID,PDB,Mutations,Ligand Chains,Receptor Chains,Ligand Name,Receptor Name,KD(M),...,Chain A (Ligand) ID,Chain B (Receptor) ID,Chain A Sequence (Ligand),Chain B Sequence (Receptor),Chain A Length,Chain B Length,Shorter Than 50 aa,Target,proteina,Y
0,0,SKEMPI v2.0,"1A22:A, B::PMID=7504735",1A22,NaN,B,A,hGH binding protein,Human growth hormone,9.000000e-10,...,B,A,FSGSEATAAILSRAPWSLQSVNPGLKTNSSKEPKFTKCRSPERETF...,FPTIPLSRLFDNAMLRAHRLHQLAFDTYQEFEEAYIPKEQKYSFLQ...,238,191,False,FPTIPLSRLFDNAMLRAHRLHQLAFDTYQEFEEAYIPKEQKYSFLQ...,FSGSEATAAILSRAPWSLQSVNPGLKTNSSKEPKFTKCRSPERETF...,-12.333171
1,1,SKEMPI v2.0,"1A4Y:A, B::PMID=9050852",1A4Y,NaN,A,B,Ribonuclease inhibitor,Angiogenin,5.000000e-16,...,A,B,SLDIQSLDIQCEELSDARWAELLPLLQQCQVVRLDDCGLTEARCKD...,QDNSRYTHFLTQHYDAKPQGRDDRYCESIMRRRGLTSPCKDINTFI...,460,123,False,QDNSRYTHFLTQHYDAKPQGRDDRYCESIMRRRGLTSPCKDINTFI...,SLDIQSLDIQCEELSDARWAELLPLLQQCQVVRLDDCGLTEARCKD...,-20.861738
2,2,SKEMPI v2.0,"1ACB:E, I::PMID=9048543",1ACB,NaN,E,I,Bovine alpha-chymotrypsin,Eglin c,1.490000e-12,...,E,I,CGVPAIQPVLSGLSRIVNGEEAVPGSWPWQVSLQDKTGFHFCGGSL...,TEFGSELKSFPEVVGKTVDQAREYFTLHYPQYDVYFLPEGSPVTLD...,245,70,False,TEFGSELKSFPEVVGKTVDQAREYFTLHYPQYDVYFLPEGSPVTLD...,CGVPAIQPVLSGLSRIVNGEEAVPGSWPWQVSLQDKTGFHFCGGSL...,-15.908478
3,4,SKEMPI v2.0,"1AK4:A, D::PMID=9223641",1AK4,NaN,A,D,Cyclophilin A,HIV-1 capsid protein,1.200000e-05,...,A,D,MVNPTVFFDIAVDGEPLGRVSFELFADKVPKTAENFRALSTGEKGF...,PIVQNLQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEG...,165,145,False,PIVQNLQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEG...,MVNPTVFFDIAVDGEPLGRVSFELFADKVPKTAENFRALSTGEKGF...,-6.709145
4,6,SKEMPI v2.0,"1B2S:A, D::PMID=7739054",1B2S,NaN,A,D,Barnase,Barstar,1.570000e-10,...,A,D,AQVINTFDGVADYLQTYHKLPDNYITASEAQALGWVASKGNLADVA...,MKKAVINGEQIRSISDLHQTLKKELALPEYYGENLDALWDCLAGWV...,110,90,False,MKKAVINGEQIRSISDLHQTLKKELALPEYYGENLDALWDCLAGWV...,AQVINTFDGVADYLQTYHKLPDNYITASEAQALGWVASKGNLADVA...,-13.367111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431,12391,PDBbind v2020,"6THG:I, J::PMID=31862858",6THG,NaN,I,J,"Cedar Virus attachment glycoprotein (G), CedVG",human ephrin-B1,4.000000e-09,...,I,J,ETGKIFCKSVSKDPDFRLKQIDYVIPVQQDRSICMNNPLLDISDGF...,ETGAKNLEPVSWSSLNPKFLSGKGLVIYPKIGDKLDIICPRAEAGR...,426,151,False,ETGAKNLEPVSWSSLNPKFLSGKGLVIYPKIGDKLDIICPRAEAGR...,ETGKIFCKSVSKDPDFRLKQIDYVIPVQQDRSICMNNPLLDISDGF...,-11.449924
6432,12402,PDBbind v2020,"6UMT:A, B::PMID=31727844",6UMT,NaN,A,B,"Programmed cell death protein 1, human PD-1 (N...","Programmed cell death 1 ligand 2, PD-L2 IgV",2.600000e-09,...,A,B,MGWSCIILFLVATATGVHSNPPTFSPALLVVTEGDSATFTCSFSST...,MIFLLLMLSLELQLHQIAALFTVTVPKELYIIEHGSDVTLECNFDT...,140,123,False,MIFLLLMLSLELQLHQIAALFTVTVPKELYIIEHGSDVTLECNFDT...,MGWSCIILFLVATATGVHSNPPTFSPALLVVTEGDSATFTCSFSST...,-11.705001
6433,12404,PDBbind v2020,"6UYS:A, B::PMID=31879127",6UYS,NaN,A,B,K37-acetylated SUMO1,phosphorylated PML-SIM,1.600000e-06,...,A,B,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...,GSGAGEAEERVVVISSSEDSDAENSSSRY,83,29,True,GSGAGEAEERVVVISSSEDSDAENSSSRY,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...,-7.902222
6434,12405,PDBbind v2020,"6UYS:C, D::PMID=31879127",6UYS,NaN,C,D,K37-acetylated SUMO1,phosphorylated PML-SIM,1.600000e-06,...,C,D,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...,GSGAGEAEERVVVISSSEDSDAENSSSRY,83,29,True,GSGAGEAEERVVVISSSEDSDAENSSSRY,GSKEGEYIKLKVIGQDSSEIHFKVKMTTHLKKLKESYAQRQGVPMN...,-7.902222
